In [3]:
# Java and Spark download / installation
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Using the stable version of Spark (3.5.4)
!wget https://archive.apache.org/dist/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz
!tar -xzf spark-3.5.4-bin-hadoop3.tgz
!pip install -q findspark

# Environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.4-bin-hadoop3"

# Pkg Apache Iceberg for Spark 3.5
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.4-bin-hadoop3/jars/

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Spark Session
spark = SparkSession.builder \
    .appName("ApacheSparkWithIcebergByVSC") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionsExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

# Create warehouse content directory
!mkdir -p /content/warehouse

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-11-02 21:59:11--  https://archive.apache.org/dist/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443..

Example: Create a catalog table and load data

In [4]:
# Drop table, if exists
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.sales")

# Create Sales table at the catalog, using iceberg
spark.sql("""
    CREATE TABLE hadoop_catalog.default.sales (
        id INT,
        product STRING,
        quantity INT,
        price DOUBLE,
        sale_date DATE
    )
    USING iceberg
""")

# Create a Spark Dataframe
data = [
    (1, "Product A", 10, 19.99, "2026-01-01"),
    (2, "Product B", 5, 29.99, "2026-01-02"),
]

columns = ["id", "product", "quantity", "price", "sale_date"]
df = spark.createDataFrame(data, columns)

# Convert the sale_date column to a date type
df = df.withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))

#Append data in the iceberg table
df.writeTo("hadoop_catalog.default.sales").append()


After that, retrieve data to confirm append command:

+---+---------+--------+-----+----------+
| id|  product|quantity|price| sale_date|
+---+---------+--------+-----+----------+
|  1|Product A|      10|19.99|2026-01-01|
|  2|Product B|       5|29.99|2026-01-02|
+---+---------+--------+-----+----------+



In [5]:
# Show data
spark.sql("SELECT * FROM hadoop_catalog.default.sales").show()

+---+---------+--------+-----+----------+
| id|  product|quantity|price| sale_date|
+---+---------+--------+-----+----------+
|  1|Product A|      10|19.99|2026-01-01|
|  2|Product B|       5|29.99|2026-01-02|
+---+---------+--------+-----+----------+



In [6]:
# Update data, and then, retrieve to confirm update command
spark.sql("""
    UPDATE hadoop_catalog.default.sales
    SET quantity = 15
    WHERE id = 1
""")

spark.sql("SELECT * FROM hadoop_catalog.default.sales WHERE id = 1").show()


+---+---------+--------+-----+----------+
| id|  product|quantity|price| sale_date|
+---+---------+--------+-----+----------+
|  1|Product A|      15|19.99|2026-01-01|
+---+---------+--------+-----+----------+



In [8]:
# Delete data, and then, retrieve all records to confirm delete command
spark.sql("""
    DELETE FROM hadoop_catalog.default.sales
    WHERE id = 2
""")

spark.sql("SELECT * FROM hadoop_catalog.default.sales").show()

+---+---------+--------+-----+----------+
| id|  product|quantity|price| sale_date|
+---+---------+--------+-----+----------+
|  1|Product A|      15|19.99|2026-01-01|
+---+---------+--------+-----+----------+



In [9]:
# Drop table, and then confirm directory files
spark.sql("DROP TABLE hadoop_catalog.default.sales")

!ls -l /content/warehouse/hadoop_catalog.db/default/sales
#

ls: cannot access '/content/warehouse/hadoop_catalog.db/default/sales': No such file or directory


Message at end of this command:
ls: cannot access '/content/warehouse/hadoop_catalog.db/default/sales': No such file or directory
